# 추세 차트 — 선 그래프 · 다중 계열 · 이동평균


**직접 풀어보는 12문제:** 정답 코드는 넣지 않았습니다. 준비 코드만 제공하며 각 문제의 빈 셀에 풀이를 작성합니다. 점검 숫자는 아래 가상 데이터에 대한 값입니다. 이 노트북의 준비 데이터에만 점검 숫자를 적용하세요.

각 노트북은 독립적으로 사용할 수 있습니다. 위에서 아래로 진행하고, 실험에는 복사본을 사용하세요. 개별 문제에서 만든 결과는 다른 이름에 저장해 원본 변수를 덮어쓰지 않는 것을 권합니다.

## 처음 배우는 분을 위한 안내

선 그래프는 순서가 있는 값의 변화를 연결합니다. 날짜형 변환 후 시간순으로 정렬해야 합니다. 이동평균은 현재 행과 앞의 여러 행을 묶은 평균입니다. window=3은 3개 관측값이며, 매일 한 행일 때 3일을 뜻합니다. 누락 날짜가 있으면 먼저 날짜 간격을 확인합니다.

**학습 순서:** 문제 1 ~ 3은 필수 기초, 4 ~ 12는 선택 도전입니다. 후반 문제는 1번의 online_daily와 3번의 월별 자료를 사용하므로 순서대로 진행합니다.

셀은 실행 단위입니다. Shift+Enter로 한 셀씩 위에서 아래로 실행합니다. 화면에 보이는 ```python``` 회색 코드 블록은 설명 문서이므로 실행하려면 새 코드 셀에 복사하세요. 준비 셀은 데이터만 만들며 문제의 답은 포함하지 않습니다. 빈 풀이 셀은 오류 없이 실행되지만 아직 결과를 계산한 것은 아닙니다. **완료할 때는 계산 결과 하나와 그 의미 한 문장을 함께 남기세요.**

그래프 창이 비어 있으면 데이터 개수와 결측 제거 결과를 먼저 확인하세요. 한글이 네모로 나오면 글꼴 설정 문제이며 계산 오류와 구분합니다.

## 실행 준비

Python 3와 `pandas`, `numpy`, `matplotlib`, `seaborn`이 필요합니다(확장 시각화 노트북은 `wordcloud`, `folium` 추가 필요). 없는 경우 별도 코드 셀에서 `%pip install pandas numpy matplotlib seaborn wordcloud folium`을 한 번 실행하세요. 이 노트북은 외부 CSV 없이 실습 데이터가 자동 생성됩니다.

Colab 또는 Jupyter에서 `.ipynb`를 열고 **준비 셀부터 순서대로** 실행하세요. 아래 코드는 한글 폰트 설정과 실습 데이터 생성만 수행합니다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# 날짜 순서가 일부 섞여 있는 일별 온라인·오프라인 가상 데이터 (10일)
online_daily_raw = pd.DataFrame({
    'order_date': [
        '2026-01-03', '2026-01-01', '2026-01-02', '2026-01-05', '2026-01-04',
        '2026-01-06', '2026-01-08', '2026-01-07', '2026-01-09', '2026-01-10'
    ],
    'orders': [9, 10, 13, 11, 14, 15, 16, 12, 18, 20],
    'revenue': [90, 100, 130, 110, 140, 150, 160, 120, 180, 210]
})

offline_daily_raw = pd.DataFrame({
    'order_date': pd.date_range('2026-01-01', periods=10).strftime('%Y-%m-%d'),
    'orders': [6, 8, 5, 9, 7, 10, 8, 11, 9, 12],
    'revenue': [70, 85, 65, 95, 80, 110, 90, 125, 105, 140]
})

# 월별·카테고리별 매출 가상 데이터 (1~6월)
monthly_cat = pd.DataFrame({
    'order_date': [
        '2026-01-15', '2026-01-20', '2026-02-10', '2026-02-18',
        '2026-03-05', '2026-03-22', '2026-04-11', '2026-04-25',
        '2026-05-09', '2026-05-19', '2026-06-14', '2026-06-28'
    ],
    'product_category': ['Tech', 'Food'] * 6,
    'revenue': [300, 180, 340, 195, 320, 210, 390, 225, 430, 240, 480, 260]
})

# 실제값 vs 예측값 비교용 데이터
eval_df = pd.DataFrame({
    'step': range(1, 11),
    'actual': [100, 256, 152, 140, 150, 183, 175, 203, 152, 198],
    'predicted': [115, 242, 176, 148, 115, 209, 185, 256, 129, 195]
})

## 문제 1. 온라인 주문 일별 매출 정렬과 기본 선 그래프 [필수 기초]

`online_daily_raw`를 복사해 `order_date`를 날짜형(`datetime`)으로 변환하고 날짜 오름차순으로 정렬한 `online_daily`를 만드세요. X축에 날짜, Y축에 `revenue`를 놓고 선 그래프(`plt.plot`)를 그리세요.

**점검 기준:** 정렬 전 첫 행은 `2026-01-03(90)`이었으나 정렬 후 `2026-01-01(100)`부터 `2026-01-10(210)`까지 시간순으로 올바르게 연결됩니다(총매출 합계 `1390`).

In [ ]:
# 문제 1 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 2. 오프라인 일별 주문 건수 선 그래프와 마커 표시 [필수 기초]

`offline_daily_raw`의 `order_date`를 날짜형으로 변환한 뒤, 일별 주문 건수(`orders`) 변화를 `marker='o'` 옵션이 있는 선 그래프로 그리세요. 제목은 `Daily Offline Orders`로 지정하세요.

**점검 기준:** 1월 1일 `6건`에서 시작해 오르내림을 반복하며 1월 10일 `12건`(최대)까지 상승하는 흐름이 원형 마커와 함께 표시됩니다(10일 총 주문 `85건`).

In [ ]:
# 문제 2 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 3. 월별 전체 매출 합계 추이 확인 [필수 기초]

`monthly_cat`의 `order_date`를 날짜형으로 변환하고 월(`dt.month`) 정보를 추출해 `month` 열을 만드세요. 월별 `revenue` 합계를 구해 시간 순서대로 선 그래프를 그리세요.

**점검 기준:** 1월 `480`, 2월 `535`, 3월 `530`, 4월 `615`, 5월 `670`, 6월 `740`으로 3월의 소폭 보합을 제외하면 전반적인 우상향 추세를 보입니다(6개월 총합 `3570`).

In [ ]:
# 문제 3 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 4. 온라인과 오프라인 일별 매출 한 차트에서 비교하기 [선택 도전]

날짜순으로 정렬된 온라인(`online_daily`)과 오프라인(`offline_daily`)의 일별 `revenue`를 하나의 그래프에 그리고, 각 선에 `Online`, `Offline` 라벨과 범례(`legend`), 제목, 축 이름을 추가하세요.

**점검 기준:** 10일 내내 `Online` 매출(합계 `1390`, 평균 `139.0`)이 `Offline` 매출(합계 `965`, 평균 `96.5`)보다 위쪽에 위치하며 두 채널 모두 우상향 추세를 보입니다.

In [ ]:
# 문제 4 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 5. 두 상품 카테고리의 월별 매출 추세 비교 [선택 도전]

`monthly_cat`에서 `Tech`와 `Food` 두 카테고리의 월별(1 ~ 6월) `revenue`를 각각 시계열로 정리해 하나의 선 그래프에 서로 다른 선 스타일(`linestyle`, `marker`)로 그리고 범례를 표시하세요.

**점검 기준:** `Tech`는 1월 `300`에서 6월 `480`으로(`+180`), `Food`는 1월 `180`에서 6월 `260`으로(`+80`) 증가하며 `Tech`의 성장 폭이 더 가파릅니다.

In [ ]:
# 문제 5 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 6. `fig, ax = plt.subplots()` OOP 방식으로 완성형 차트 만들기 [선택 도전]

`online_daily`의 날짜별 매출을 `fig, ax = plt.subplots(figsize=(8, 4))` 방식으로 그리세요. 라벨은 `Daily Revenue`, 선 두께 `linewidth=2`, 마커 `'o'`를 지정하고 `ax.set_xlabel`, `ax.set_ylabel`, `ax.set_title`, `ax.legend()`, `ax.grid(True, alpha=0.3)`을 모두 적용하세요.

**점검 기준:** 객체지향(`ax.set_*`) 메서드로 제목·축·범례·격자가 모두 포함된 표준 보고서용 선 그래프가 완성됩니다.

In [ ]:
# 문제 6 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 7. 실제값 · 예측값 · 오차 다중 계열 선 그래프 [선택 도전]

`eval_df`에서 `error = actual - predicted` 열을 계산한 뒤, `step`을 X축으로 하여 `actual`(녹색 실선), `predicted`(빨간 파선), `error`(파란 점선) 세 계열을 한 차트에 그리고 범례를 표시하세요.

**점검 기준:** `error`의 최댓값은 `step=5`에서의 `+35(150 - 115)`, 최솟값(최대 과대예측)은 `step=8`에서의 `-53(203 - 256)`입니다.

In [ ]:
# 문제 7 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 8. 축 범위(`xlim`, `ylim`) 조정과 최고점 주석(`annotate`) 추가 [선택 도전]

`online_daily` 매출 선 그래프에서 Y축 범위를 `50`부터 `250`까지(`ylim(50, 250)`)로 지정하고, 매출이 가장 높은 날짜(`2026-01-10`, 매출 `210`) 위치에 `plt.annotate()`로 화살표와 `'최고 매출 (210)'` 텍스트를 표시하세요.

**점검 기준:** Y축 여백이 확보되어 화살표 주석이 잘리지 않고 최고점 `(2026-01-10, 210)`을 정확히 가리킵니다.

In [ ]:
# 문제 8 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 9. 오프라인 주문 건수의 3일 이동평균과 초기 `NaN` 확인 [선택 도전]

`offline_daily`의 `orders`에 대해 `rolling(window=3).mean()`으로 3일 이동평균 열 `ma3`를 만들고, 앞의 3개 행을 출력해 `NaN` 개수와 3번째 날짜(`2026-01-03`)의 이동평균 값을 확인한 뒤 원본 선과 함께 그리세요.

**점검 기준:** 첫 2개 행(1월 1일, 1월 2일)은 `NaN`이며, 1월 3일의 3일 이동평균은 `(6 + 8 + 5) / 3 = 6.3333...`(약 `6.33`)입니다.

In [ ]:
# 문제 9 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 10. 온라인 일별 매출의 5일 이동평균 계산 및 추세 비교 [선택 도전]

`online_daily`의 `revenue`에 대해 `rolling(window=5).mean()`으로 5일 이동평균 `ma5`를 구하고, 원본 매출(`alpha=0.4`)과 5일 이동평균(`linewidth=2.5`)을 한 그래프에 겹쳐 그리세요.

**점검 기준:** 앞 4개 행이 `NaN`이고 5번째 날(`2026-01-05`)의 첫 5일 이동평균은 `(100+130+90+140+110)/5 = 114.0`, 마지막 날(`2026-01-10`)은 `(150+120+160+180+210)/5 = 164.0`입니다.

In [ ]:
# 문제 10 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 11. 이동평균 `window` 크기(3일 vs 5일)에 따른 곡선 비교 [선택 도전]

`online_daily`의 원본 `revenue`, 3일 이동평균(`ma3`), 5일 이동평균(`ma5`) 세 개의 선을 하나의 차트에 그리고 어느 선이 더 부드러운지, 그리고 `window`가 커질 때의 장단점을 적으세요.

**점검 기준:** 1월 3일의 첫 `ma3`는 `(100+130+90)/3 = 106.67`, 1월 10일의 `ma3`는 `(160+180+210)/3 = 183.33`입니다. `ma5`가 `ma3`보다 일별 출렁임이 더 완만하지만 시작 시점이 2일 더 늦습니다.

In [ ]:
# 문제 11 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 문제 12. `2×2 Subplots` 추세 대시보드 구성하기 [선택 도전]

`fig, axes = plt.subplots(2, 2, figsize=(11, 7))`를 사용해 ①(좌상) 온라인·오프라인 일별 매출 비교, ②(우상) 온라인 매출과 3일·5일 이동평균, ③(좌하) 카테고리별 월별 매출, ④(우하) `eval_df`의 실제값과 예측값 비교를 한 화면에 배치하고 `plt.tight_layout()`을 적용하세요.

**점검 기준:** 4개의 하위 차트가 겹침 없이 한 화면에 정렬되어 일별·월별·이동평균·예측 오차 추이를 종합 비교할 수 있습니다.

In [ ]:
# 문제 12 풀이를 작성하세요.

**결과 해석 / 근거 / 새로 알게 된 점:**

여기에 작성하세요.

## 선택 도전: 같은 날짜 주문을 먼저 합치기

**필수 아님:** 위의 기본 문제를 먼저 완료하세요. 새 파일을 구하거나 준비 셀의 변수를 교체하지 않습니다. 아래 작은 입력만 별도 코드 셀에 복사해 사용합니다.

### 입력

```python
import pandas as pd
extra_orders = pd.DataFrame({'date':['2026-01-01','2026-01-01','2026-01-02'],'revenue':[10,20,40]})
```

### 진행 순서

1. date를 pd.to_datetime으로 날짜형으로 바꿉니다.
2. 날짜별 revenue 합계를 구하고 날짜순으로 정렬합니다.
3. 일별 합계로 선 그래프를 그립니다. 주문별 3행을 바로 연결한 경우와 무엇이 다른지 적습니다.

**완료 기준:** 1월1일 30, 1월2일 40의 두 점. 일별 합계 70은 주문 합계 70과 같음.

## 제출 전 확인

- 준비 셀과 풀이를 위에서 아래로 실행했나요?
- 그래프에 제목(`title`), 축 이름(`xlabel`, `ylabel`), 다중 계열인 경우 범례(`legend`)를 표시했나요?
- 모든 점검 기준에 대해 수치 근거와 시각적 특징을 함께 남겼나요?
- 결측값 처리 여부와 표본 크기(또는 구간/윈도우 설정)를 확인했나요?
- 원본 데이터프레임 변수를 덮어쓰지 않고 안전하게 실습했나요?